# DS-04

Streaming Algorithms

**Web page:** <a href="https://apagyidavid.web.elte.hu/2025-2026-2/ds"
target="_blank">apagyidavid.web.elte.hu/2025-2026-2/ds</a>

<a target="_blank" href="https://colab.research.google.com/github/dapagyi/ds-web/blob/notebooks/ds-04.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Flajolet Martin Algorithm**: estimate number of distinct elements seen
so far in a data stream

*In particular*: gives good estimate using very little memory

Examples to keep in mind: web clicks, sensor data; storing every element
in memory is expensive

**Algorithm**:

1.  Hash elements uniformly to binary strings
2.  Count number of trailing zeros in binary hash
3.  Store maximum number of trailing zeros seen so far: R
4.  Estimate number of unique elements $\sim 2^R$

**Key Observations**:

- If you hash elements using a good uniform hash function, then
  probability of r trailing 0s decays as $1/2^r$
- You are unlikely to see a hash that ends in r trailing zeros unless
  you’ve seen $\sim 2^r$ distinct elements!

**Mathematically**: $\max(r_1,...,r_n) \sim \log_2(n)$

Heuristic Example: If I flip a coin 100 times, that generates a 100 bit
binary string. Say I do this 5 times–I would be surprised if I flipped,
say, 60 heads in a row.

But if I did this 1,000,000,000 times, I would be less surprised if one
outcome had 60 leading heads.

------------------------------------------------------------------------

**A Few Technicalities for Coding**:

1.  Choosing the Bit-Length:

- We need enough bits for accuracy: we expect to see about $\log_2(n)$
  trailing 0s, where n is the number of distinct items

ex: If estimating up to a billion unique elements, we might see up to
$\log_2(10^9) \sim 30$ trailing 0s. So in this case, the hash needs at
least 30 bits of randomness

- We also want to avoid collisions between distinct samples, which would
  bias the estimate

*In our example below*: we use 160 bits

1.  Multiple Hash Functions: the estimate will be noisy, which we reduce
    by running multiple parallel estimators using the same data but
    independent hash functions. This averages out the randomness.

In practice, we do this by appending a random string to each element
(fixed within an iteration) and hashing those in parallel using the same
hash function.

*This is a cheap simulation of using multiple independent hash
functions*

1.  Implementation of Hash Function: we use SHA (Secure Hash Algorithm),
    which is a family of cryptographic hash functions designed by the
    NSA and standardized by NIST

- Appear random but are deterministic
- We use SHA-1, which has bit size 160, and is fast but not secure
  (collisions found)
- This is fine for our purposes as we care about speed and uniformity
  (alternative for bigger examples: SHA-256)

**Not Recommended**: built-in `hash()` from Python because not
consistent across runs (random seed used for security)

In [1]:
#Here's a script implementing the Flajolet-Martin Algorithm in Python for a simple example

import hashlib
import random
import numpy as np

def hash_item(item, seed=0):
    random.seed(seed) #ensures deterministic behavior per seed
    salt = random.randint(0, 1 << 31) #changes the hash function for each seed
    item_str = str(item) + str(salt)
    #print(item_str)
    hash_val = hashlib.sha1(item_str.encode()).hexdigest() #returns in hexadecimal string rep--compact, easy to convert to string and/or binary
    #print(hash_val)
    bin_val = bin(int(hash_val, 16))[2:]
    #print(bin_val)
    #breakpoint()
    return bin_val.zfill(160) # Ensure fixed length

def trailing_zeros(bin_str):
    return len(bin_str) - len(bin_str.rstrip('0'))

def flajolet_martin(stream, num_hashes=16):
    max_trailing = [0] * num_hashes
    
    for item in stream:
        for i in range(num_hashes):
            h = hash_item(item, seed=i)
            tz = trailing_zeros(h)
            max_trailing[i] = max(max_trailing[i], tz)
            
    estimates = [2 ** r for r in max_trailing]
    return int(np.median(estimates)) # Robust to outliers

# Example stream with duplicates
stream=['apple', 'banana', 'apple', 'orange', 'banana', 'grape', 'melon', 'apple']
print("Estimated unique elements:", flajolet_martin(stream))

Estimated unique elements: 6

**Improvements on Flajolet-Martin**: PySpark ML doesn’t natively include
Flajolet-Martin, but has an equivalent (and stronger) implementation
called approxCountDistinct

- Based on HyperLogLog: whereas FM is analogous to flipping many coins,
  noting longest run of heads, and estimating rarity; HyperLogLog is
  like organizing many buckets that *each* tracking longest run of heads
  and then combining them intelligently

| Feature | Flajolet-Martin (FM) | HyperLogLog (HLL) |
|:-----------------------|:-----------------------|:-----------------------|
| **Origin** | Introduced in 1983 by Flajolet & Martin | Introduced in 2007 (based on FM & LogLog) |
| **Goal** | Estimate number of distinct items | Same, but with **higher accuracy** and **less memory** |
| **Basic Idea** | Track **max trailing zeros** in hash values | Use multiple buckets, track **max trailing zeros per bucket** |
| **Accuracy** | Low to moderate | Much higher (error $\approx 1.04/\sqrt{m}$, where m = buckets) |
| **Multiple Estimators** | Simulated via multiple hash functions | Realized by splitting hash space into **buckets** |
| **Memory Usage** | Very low (~bits) | Low (~1-2 KB for good accuracy) |
| **Combining Estimates** | Average or median of multiple estimators | Harmonic mean with bias correction |
| **Bias Correction** | None (or very basic) | Yes - empirically derived corrections |
| **Scalability** | Good, but primitive | Excellent - production-grade (used in Redis, Spark, etc.) |
| **Use in Practice** | Educational, foundation | Industrial use (Google BigQuery, AWS, Redis, etc.) |

In [2]:
# This approx counting algorithm is supported natively in Spark for both SQL and DataFrame functions
# We look at a DataFrame example (for import options, see previous week notebook)

from pyspark.sql import SparkSession
from pyspark.sql.functions import approx_count_distinct

# Create Spark session
spark = SparkSession.builder.appName("ApproxCountDistinct").getOrCreate()

# Example data with duplicates
data = [("apple",), ("banana",), ("apple",), ("orange",),
        ("banana",), ("grape",), ("melon",), ("apple",)]

# Create DataFrame
df = spark.createDataFrame(data, ["fruit"])

# Use approx_count_distinct
df.select(approx_count_distinct("fruit").alias("estimated_unique")).show()

#There is also an option to specify relative standard deviation (rsd)
#default is ~95% confidence, rsd=0.05

#Smaller rsd ---> better estimate, but more memory
df.select(approx_count_distinct('fruit', rsd=0.3).alias("estimated_unique")).show()

+----------------+
|estimated_unique|
+----------------+
|               5|
+----------------+

+----------------+
|estimated_unique|
+----------------+
|               6|
+----------------+


**AMS Method**: Alon-Matias-Szegedy, designed to estimate *frequency
moments* of data stream using very little memory

**Review of Method**: Let $a_1, \dots, a_n$ be the stream elements,
where $a_i$ has frequency $f_i$. Then the $k^{th}$ *frequency moment* is
$$F_k = \sum_i f_i^k$$

| Moment | Meaning | Use |
|:-----------------------|:-----------------------|:-----------------------|
| $F_0$ | Number of **distinct elements** | Cardinality estimation |
| $F_1$ | **Total number of elements** | Just count - trivial to compute |
| $F_2$ | **Sum of squares of frequencies** | Used to detect skew / duplicates |
| $F_k \ge 3$ | Higher moments - more complex | Useful in advanced statistics |

**Note**: $F_0$ is what we estimated above.

**How we look at AMS for second moment**

*What is the second moment measuring?* The second moment measures
*skew*–how often the elements repeat

- Low $F_2 \rightarrow$ values are uniformly distributed
- High $F_2 \rightarrow$ skewed, with duplicates dominating

**Goal of AMS**: use an incremental estimator using randomized sampling
from the stream.

1.  **Choose random time t**: sample an element from the stream, which
    will be the *representative element*
2.  **Track Counts**:
    - X.el: keep track of sampled element $X_{el}$, the element chosen
      at time $t$
    - X.val: track the **count** of how many times this element $X_{el}$
      has appeared in the stream after time $t$.
3.  **Estimate** $$S = f(X_{el}) = n(2c - 1)$$ , where $c$ is the count
    of $X_{el}$ and $n$ is the length of the stream.
4.  **Repeat at multiple randomly sampled times**

**Recall from Lecture**: in expectation, this estimate gives the 2nd
moment, converging as we take more samples.

In [3]:
#[4] #Let's check it out in Python!

import random

# Stream simulation
stream = ['a', 'b', 'a', 'c', 'a', 'b', 'd', 'e', 'a', 'b']

# Define the function that simulates the AMS approach with multiple X's
def ams_second_moment(stream, num_estimators=10):
    n = len(stream)  # Total number of elements in the stream
    estimates = []
    
    for _ in range(num_estimators):
        # Randomly choose the start time t
        t = random.randint(0, n-1)
        
        # Track the element X.el and its count X.val after time t
        X_el = stream[t]  # The element at time t
        X_val = 0
        for i in range(t, n):
            if stream[i] == X_el:
                X_val += 1
                
        # Calculate the estimate for this X
        S = n * (2 * X_val - 1)
        estimates.append(S)
        
    # The final estimate for F2 is the average of all estimates
    return sum(estimates) / len(estimates)

# True F2 (For comparison)
from collections import defaultdict
true_f2 = sum(f**2 for f in defaultdict(int, {e: stream.count(e) for e in set(stream)}).values())

# Run the AMS estimator
estimated_f2 = ams_second_moment(stream, num_estimators=1)
print(f"True F2: {true_f2}")
print()
print(f"Estimated F2 with 1 estimator: {estimated_f2}")
print(f"Error with 1 estimator: {estimated_f2-true_f2}")

print()
estimated_f2 = ams_second_moment(stream, num_estimators=4)
print(f"Estimated F2 with 4 estimators: {estimated_f2}")
print(f"Error with 4 estimators: {estimated_f2-true_f2}")

print()
estimated_f2 = ams_second_moment(stream, num_estimators=7)
print(f"Estimated F2 with 7 estimators: {estimated_f2}")
print(f"Error with 7 estimators: {estimated_f2-true_f2}")

print()
estimated_f2 = ams_second_moment(stream, num_estimators=10)
print(f"Estimated F2 with 10 estimators: {estimated_f2}")
print(f"Error with 10 estimators: {estimated_f2-true_f2}")

print()
estimated_f2 = ams_second_moment(stream, num_estimators=100)
print(f"Estimated F2 with 100 estimators: {estimated_f2}")
print(f"Error with 100 estimators: {estimated_f2-true_f2}")

print()
estimated_f2 = ams_second_moment(stream, num_estimators=1000)
print(f"Estimated F2 with 1000 estimators: {estimated_f2}")
print(f"Error with 1000 estimators: {estimated_f2-true_f2}")

True F2: 28

Estimated F2 with 1 estimator: 70.0
Error with 1 estimator: 42.0

Estimated F2 with 4 estimators: 35.0
Error with 4 estimators: 7.0

Estimated F2 with 7 estimators: 44.285714285714285
Error with 7 estimators: 16.285714285714285

Estimated F2 with 10 estimators: 22.0
Error with 10 estimators: -6.0

Estimated F2 with 100 estimators: 27.8
Error with 100 estimators: -0.1999999999999993

Estimated F2 with 1000 estimators: 27.36
Error with 1000 estimators: -0.6400000000000006

We see that increasing the number of estimators generally improves the
estimate, but there is quite a lot of variance (try running the above
cell a few times).

General guidance for choosing number of estimators: for error threshold
$\varepsilon$, choose number of estimators at least $1/\varepsilon^2$.

| Desired Relative Error | Minimum $k$ |
|:-----------------------|:------------|
| 30%                    | ~11         |
| 10%                    | 100         |
| 5%                     | 400         |
| 1%                     | 10,000      |